In [8]:
import os
import pandas as pd
import numpy as np

DATA_DIR = r"C:\Users\diksh\OneDrive\Desktop\RailWise\SIH_26027_Final_Dataset"

task_filename = None

for file in os.listdir(DATA_DIR):
    if file.lower().endswith(".csv"):
        path = os.path.join(DATA_DIR, file)
        temp = pd.read_csv(path, nrows=5)

        if "maintenance_priority_score" in temp.columns:
            task_filename = file
            break

print("Task file found:", task_filename)

tasks = pd.read_csv(
    os.path.join(DATA_DIR, task_filename)
)

print("Dataset shape:", tasks.shape)


Task file found: unified_maintenance.csv
Dataset shape: (14400, 21)


In [10]:
target = "maintenance_priority_score"

print(tasks[target].describe())

print("\nMissing values:")
print(tasks[target].isna().sum())

print("\nUnique values:")
print(tasks[target].nunique())

count    14400.000000
mean        42.877312
std         14.512320
min         13.500000
25%         31.400000
50%         40.600000
75%         52.900000
max         94.400000
Name: maintenance_priority_score, dtype: float64

Missing values:
0

Unique values:
679


In [11]:
numeric_cols = tasks.select_dtypes(
    include=np.number
).columns

correlations = (
    tasks[numeric_cols]
    .corr()[target]
    .sort_values(ascending=False)
)

print(correlations)

maintenance_priority_score    1.000000
risk_score                    0.835432
safety_risk_1_5               0.792848
urgency_score                 0.772157
operational_impact_1_5        0.608575
overdue_days                  0.579137
criticality_1_5               0.423386
asset_downtime_risk           0.212583
estimated_duration_min        0.120548
required_team_size            0.045942
location_km                   0.003930
Name: maintenance_priority_score, dtype: float64


In [12]:
features = [
    "department",
    "asset_type",
    "corridor_id",
    "location_km",
    "criticality_1_5",
    "safety_risk_1_5",
    "operational_impact_1_5",
    "overdue_days",
    "estimated_duration_min",
    "required_team_size",
    "possession_required",
    "maintenance_type",
    "severity"
]

X = tasks[features].copy()
y = tasks[target].copy()

print("Features:", X.shape)
print("Target:", y.shape)

display(X.head())

Features: (14400, 13)
Target: (14400,)


,department,asset_type,corridor_id,location_km,criticality_1_5,safety_risk_1_5,operational_impact_1_5,overdue_days,estimated_duration_min,required_team_size,possession_required,maintenance_type,severity
0,Engineering,Weld,C00069,127.8,4,2,2,19,75,2,Yes,Preventive,Medium
1,Engineering,Ballast,C00033,0.3,2,1,2,0,43,3,Yes,Preventive,Low
2,Engineering,Crossing,C00078,7.6,2,3,2,56,91,4,No,Preventive,Medium
3,Engineering,Sleeper,C00002,30.1,2,3,1,109,114,2,Yes,Preventive,Medium
4,Engineering,Crossing,C00078,24.7,2,5,3,44,78,2,Yes,Preventive,High


In [13]:
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())

for col in X.select_dtypes(exclude=np.number).columns:
    X[col] = X[col].fillna("Unknown")

print("Remaining missing values:", X.isnull().sum().sum())

Remaining missing values: 0


In [14]:
X = pd.get_dummies(
    X,
    columns=[
        "department",
        "asset_type",
        "corridor_id",
        "possession_required",
        "maintenance_type",
        "severity"
    ],
    drop_first=True
)

print("Final feature shape:", X.shape)

Final feature shape: (14400, 140)


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (11520, 140)
Testing: (2880, 140)
